# Walkthrough Viewer

This notebook provides a way to visualize the generated walkthrough with:
1. Separated blocks
2. Component link visualization
3. Interactive exploration of the walkthrough structure

In [3]:
import json
import re
from IPython.display import display, HTML, Markdown
from ipywidgets import interact, widgets, Layout, VBox, HBox, Button, Output
import pandas as pd

In [4]:
# Load the walkthrough data
with open('tiny_demo_walkthrough.json', 'r') as f:
    walkthrough_data = json.load(f)

print(f"Loaded walkthrough with {len(walkthrough_data['sections'])} sections")
print(f"Total components: {walkthrough_data.get('total_components', 'N/A')}")
print(f"Components covered: {walkthrough_data.get('components_covered', 'N/A')}")

Loaded walkthrough with 5 sections
Total components: 19
Components covered: 19


## 1. Block Overview

In [5]:
# Display block overview
blocks_df = pd.DataFrame([
    {
        'Block ID': section['block_id'],
        'Block Name': section['block_name'],
        'Components Referenced': len(section['components_referenced'])
    }
    for section in walkthrough_data['sections']
])

display(blocks_df)

,Block ID,Block Name,Components Referenced
0,1,Import Libraries,8
1,2,Text Cleaning and Tokenization Functions,7
2,3,VectorStore Class Definition,15
3,4,Build RAG Graph Function,11
4,5,Configuration Variables,4


## 2. Interactive Block Viewer

In [ ]:
def highlight_component_links(content):
    """Convert component links to HTML with styling"""
    # Pattern to match component links
    pattern = r'\[\[component:(\d+):(\d+):([^|]+)\|([^\]]+)\]\]'
    
    def replace_link(match):
        block_id = match.group(1)
        comp_num = match.group(2)
        comp_name = match.group(3)
        display_text = match.group(4)
        
        # Create styled HTML link
        return (f'<span style="background-color: #FFEB3B; padding: 2px 4px; '
                f'border-radius: 3px; font-weight: bold; cursor: pointer;" '
                f'title="Block {block_id}, Component {comp_num}: {comp_name}">'
                f'{display_text}</span>')
    
    # Replace all component links with styled spans
    html_content = re.sub(pattern, replace_link, content)
    
    # Convert markdown formatting
    html_content = html_content.replace('**', '<strong>').replace('**', '</strong>')
    html_content = html_content.replace('\n\n', '</p><p>').replace('\n', '<br>')
    html_content = f'<p>{html_content}</p>'
    
    return html_content

# Create interactive viewer
def view_block(block_index):
    section = walkthrough_data['sections'][block_index]
    
    # Create styled HTML output
    html = f"""
    <div style="border: 2px solid #2196F3; border-radius: 8px; padding: 20px; margin: 10px 0;">
        <h2 style="color: #2196F3; margin-top: 0;">Block {section['block_id']}: {section['block_name']}</h2>
        <div style="background-color: #f5f5f5; padding: 15px; border-radius: 5px; margin-bottom: 15px;">
            {highlight_component_links(section['content'])}
        </div>
        <div style="margin-top: 20px;">
            <h3>Components Referenced ({len(section['components_referenced'])})</h3>
            <ul style="list-style-type: none; padding-left: 0;">
    """
    
    for comp in section['components_referenced']:
        html += f"""
            <li style="margin: 5px 0; padding: 5px; background-color: #e3f2fd; border-radius: 3px;">
                <strong>Block {comp['block_id']}, Component {comp['component_number']}:</strong> 
                {comp['component_name']} → "{comp['display_text']}"
            </li>
        """
    
    html += """
            </ul>
        </div>
    </div>
    """
    
    display(HTML(html))

# Create dropdown for block selection
block_options = [(f"Block {s['block_id']}: {s['block_name']}", i) 
                 for i, s in enumerate(walkthrough_data['sections'])]

interact(view_block, block_index=widgets.Dropdown(
    options=block_options,
    description='Select Block:',
    style={'description_width': 'initial'},
    layout=Layout(width='400px')
))

interactive(children=(Dropdown(description='Select Block:', layout=Layout(width='400px'), options=(('Block 1: …

<function __main__.view_block(block_index)>

## 3. Component Cross-Reference Analysis

In [7]:
# Analyze component cross-references
component_refs = {}

for section in walkthrough_data['sections']:
    for comp in section['components_referenced']:
        comp_key = f"{comp['component_name']} (Block {comp['block_id']})"
        if comp_key not in component_refs:
            component_refs[comp_key] = []
        component_refs[comp_key].append(f"Block {section['block_id']}")

# Display components that are referenced across multiple blocks
cross_refs = {k: v for k, v in component_refs.items() if len(set(v)) > 1}

if cross_refs:
    print("Components referenced across multiple blocks:")
    for comp, refs in sorted(cross_refs.items()):
        print(f"\n{comp}:")
        print(f"  Referenced in: {', '.join(sorted(set(refs)))}")
else:
    print("No components are referenced across multiple blocks.")

Components referenced across multiple blocks:

VectorStore (Block 3):
  Referenced in: Block 1, Block 3

build_rag_graph (Block 4):
  Referenced in: Block 3, Block 4

clean_text (Block 2):
  Referenced in: Block 1, Block 2, Block 3, Block 4

tokenize (Block 2):
  Referenced in: Block 2, Block 4


## 4. Full Walkthrough with Visual Separation

In [8]:
# Display full walkthrough with visual separation
html_output = f"""
<div style="max-width: 900px; margin: 0 auto;">
    <h1 style="color: #1976D2; text-align: center; margin-bottom: 20px;">Code Walkthrough</h1>
    <div style="background-color: #E3F2FD; padding: 15px; border-radius: 8px; margin-bottom: 30px;">
        <p style="font-size: 16px; line-height: 1.6; margin: 0;">{walkthrough_data['introduction']}</p>
    </div>
"""

for i, section in enumerate(walkthrough_data['sections']):
    # Add visual separator between blocks
    if i > 0:
        html_output += '<hr style="border: 2px solid #E0E0E0; margin: 40px 0;">'
    
    html_output += f"""
    <div style="background-color: #FAFAFA; border-left: 4px solid #2196F3; padding: 20px; margin: 20px 0;">
        <h2 style="color: #1976D2; margin-top: 0;">Block {section['block_id']}: {section['block_name']}</h2>
        <div style="line-height: 1.8;">
            {highlight_component_links(section['content'])}
        </div>
    </div>
    """

html_output += "</div>"

# Create a scrollable output
output = Output(layout=Layout(height='600px', overflow='auto'))
with output:
    display(HTML(html_output))
display(output)

Output(layout=Layout(height='600px', overflow='auto'))

## 5. Export Options

In [9]:
# Function to generate standalone HTML
def generate_standalone_html():
    html = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Code Walkthrough</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            line-height: 1.6;
            max-width: 900px;
            margin: 0 auto;
            padding: 20px;
            background-color: #f5f5f5;
        }}
        .component-link {{
            background-color: #FFEB3B;
            padding: 2px 6px;
            border-radius: 3px;
            font-weight: bold;
            cursor: pointer;
            transition: background-color 0.3s;
        }}
        .component-link:hover {{
            background-color: #FDD835;
        }}
        .block-section {{
            background-color: white;
            border-left: 4px solid #2196F3;
            padding: 20px;
            margin: 20px 0;
            box-shadow: 0 2px 5px rgba(0,0,0,0.1);
        }}
        h1, h2 {{
            color: #1976D2;
        }}
        .intro {{
            background-color: #E3F2FD;
            padding: 20px;
            border-radius: 8px;
            margin-bottom: 30px;
        }}
    </style>
</head>
<body>
    <h1>Code Walkthrough</h1>
    <div class="intro">
        <p>{walkthrough_data['introduction']}</p>
    </div>
"""
    
    for section in walkthrough_data['sections']:
        # Convert component links to HTML
        content_html = section['content']
        pattern = r'\[\[component:(\d+):(\d+):([^|]+)\|([^\]]+)\]\]'
        content_html = re.sub(pattern, 
                            r'<span class="component-link" title="Block \1, Component \2: \3">\4</span>', 
                            content_html)
        
        # Convert markdown formatting
        content_html = content_html.replace('**', '<strong>').replace('**', '</strong>')
        content_html = content_html.replace('\n\n', '</p><p>').replace('\n', '<br>')
        
        html += f"""
    <div class="block-section">
        <h2>Block {section['block_id']}: {section['block_name']}</h2>
        <p>{content_html}</p>
    </div>
        """
    
    html += """
</body>
</html>
    """
    
    with open('walkthrough_viewer.html', 'w') as f:
        f.write(html)
    
    print("Standalone HTML file created: walkthrough_viewer.html")

# Generate the HTML file
generate_standalone_html()

# Display download button
from IPython.display import FileLink
display(FileLink('walkthrough_viewer.html'))

Standalone HTML file created: walkthrough_viewer.html


/home/aesyr/repos/ember_extension/ember/walkthrough_viewer.html

## 6. Component Link Testing

In [10]:
# Test component link parsing
def parse_component_links(text):
    """Parse component links from walkthrough text"""
    pattern = r'\[\[component:(\w+):(\d+):([^|]+)\|([^\]]+)\]\]'
    matches = re.findall(pattern, text)
    
    links = []
    for match in matches:
        links.append({
            "block_id": match[0],
            "component_number": int(match[1]),
            "component_name": match[2],
            "display_text": match[3],
            "full_link": f"[[component:{match[0]}:{match[1]}:{match[2]}|{match[3]}]]"
        })
    
    return links

# Test with the first section
test_section = walkthrough_data['sections'][0]
parsed_links = parse_component_links(test_section['content'])

print(f"Testing component link parsing on Block {test_section['block_id']}:")
print(f"Found {len(parsed_links)} component links\n")

# Display first 3 links as examples
for i, link in enumerate(parsed_links[:3]):
    print(f"Link {i+1}:")
    print(f"  Full: {link['full_link']}")
    print(f"  Block: {link['block_id']}, Component: {link['component_number']}")
    print(f"  Name: {link['component_name']}")
    print(f"  Display: {link['display_text']}")
    print()

Testing component link parsing on Block 1:
Found 8 component links

Link 1:
  Full: [[component:2:1:clean_text|Text Cleaning and Tokenization Functions]]
  Block: 2, Component: 1
  Name: clean_text
  Display: Text Cleaning and Tokenization Functions

Link 2:
  Full: [[component:1:1:Import_pandas_library|Import pandas library]]
  Block: 1, Component: 1
  Name: Import_pandas_library
  Display: Import pandas library

Link 3:
  Full: [[component:2:1:clean_text|Text Cleaning and Tokenization Functions]]
  Block: 2, Component: 1
  Name: clean_text
  Display: Text Cleaning and Tokenization Functions

